# A roof, a main space and a heated basement

This illustrative model retains a 20 °C main programme, an 18 °C basement and a declared 0.5 ACH attic exchange. Core owns the physical geometry and controls; RC uses two coupled conditioned domains and an explicitly selected finite attic buffer. The packaged weather and acceptance checks are software examples, not EPC calibration.


In [ ]:
from importlib.resources import files
import numpy as np
from epsm_core import (
    Building, Footprint, GeometrySpec, RoofSpec, ScheduleSpec,
    UnconditionedExchangeSpec, Weather, ZoneUseSpec,
)
from epsm_rc import RCSettings, simulate_building

building = Building(
    preset="illustrative-reference",
    geometry=GeometrySpec.from_footprint(
        Footprint.rectangle(10, 8), storeys=3, storey_height_m=3,
        roof=RoofSpec(shape="gable", pitch_deg=30),
        basement_depth_m=2.5, basement_conditioned=True,
    ),
    unconditioned_exchange=(UnconditionedExchangeSpec(
        zone_kind="attic", outdoor_air_ach=0.5,
        source="illustrative declared attic exchange",
    ),),
)
building.operation = building.operation.updated(
    heating_setpoint=ScheduleSpec.constant(20),
)
building.zone_uses = (ZoneUseSpec(
    zone_kind="basement",
    use=building.operation.updated(heating_setpoint=ScheduleSpec.constant(18)),
    source="illustrative declared basement control",
),)
building.readiness(engine="epsm_rc").require()


The ground multiplier and sensible-only airflow policy are explicit approximations. The attic buffer keeps finite roof/ventilation resistance and shell solar/sky forcing; it omits attic air and shell storage. The physical separating floor exchanges equal and opposite heat between main space and basement.


In [ ]:
weather = Weather.from_epw(files("epsm_rc").joinpath("data/epw/goteborg.epw"))
settings = RCSettings.from_policy(
    "epsm_rc_5r1c_v2",
    airflow_policy="sensible_only_unlimited_supply",
    ground_policy="conductance_multiplier",
    cold_attic_policy="quasi_steady_buffer",
)
snapshot = building.snapshot()
result = simulate_building(building, weather=weather, settings=settings)
assert building.snapshot().sha256 == snapshot.sha256
assert result.diagnostics["accepted"]
assert result.diagnostics["domain_reduction"]["maximum_interzone_exchange_sum_w"] == 0
assert result.diagnostics["attic_reduction"]["effective_outdoor_air_ach"] == 0.5
np.testing.assert_array_equal(
    result.hourly["interzone_heat_to_main_w"],
    -result.hourly["interzone_heat_to_basement_w"],
)
np.testing.assert_allclose(
    result.hourly["space_heating_demand_kwh"],
    result.hourly["main_space_heating_demand_kwh"]
    + result.hourly["basement_space_heating_demand_kwh"],
)


In [ ]:
{
    "annual_space_heat_kwh": round(result.scalars["space_heating_demand_kwh"]),
    "conditioned_domains": [
        {"name": domain["name"], "area_m2": domain["area_m2"],
         "heating_schedule": domain["use"]["heating_setpoint"]}
        for domain in result.diagnostics["domain_reduction"]["domains"]
    ],
    "floor_conductance_w_k": result.diagnostics["domain_reduction"]["interzone_floor_conductance_w_k"],
    "attic": result.diagnostics["attic_reduction"],
}


Separate hourly temperature and heat-demand channels remain available for each domain. The top-level temperature is a volume-weighted reporting value. Schools, mixed uses and unheated basements need their own supported physical/control reduction; this example does not assign a basement's programme from age or building type. See [the reduction assumptions](../docs/roofs-and-basements.md).
